# Twisted Parallel Particle Filter — 1D Linear-Gaussian SSM (Julia)

This notebook is a line-by-line Julia translation of the Python/JAX version. The algorithm is identical; only the language idioms differ.

## The problem

We observe a noisy time series $y_{1:T}$ and want to recover the hidden states $x_{1:T}$. The model is:
$$x_0 \sim \mathcal{N}(0,1), \qquad x_t = \phi\,x_{t-1} + \sigma_q\,\varepsilon_t, \qquad y_t = x_t + \sigma_r\,\eta_t$$

Because the model is linear and Gaussian, the exact filtering posterior $p(x_t\mid y_{1:t})$ is available from the Kalman filter. We use it as ground truth to evaluate the particle filter.

## What the algorithm does

A standard particle filter is sequential: each step depends on the previous one, so $O(T)$ steps must run in order. This notebook demonstrates two improvements:

1. **Twisted proposals** — instead of sampling from the blind prior $p(x_t\mid x_{t-1})$, we multiply by a *twist function* $\psi_t(x_t)$ that anticipates future observations. A well-chosen twist keeps particle weights near-uniform, eliminating the need for resampling. Without resampling, each particle trajectory is an independent deterministic recurrence.

2. **Parallel prefix scan** — because each trajectory is a deterministic function of its initial state and pre-drawn noise, we can solve all $T$ steps simultaneously using an associative prefix scan in $O(\log T)$ parallel depth (analogous to JAX's `lax.associative_scan`).

For $D=1$ with a Gaussian twist the step function is affine in $x_{t-1}$, so the scan is exact (one Newton step suffices).

## Roadmap

| Step | What |
|------|------|
| 1 | Kalman filter — exact posterior and twist initialisation |
| 2 | `twisted_step` and `log_weight` — proposal and correction |
| 3 | Simulate ground-truth data |
| 4 | Run Kalman filter on simulated data |
| 5 | Build the noise tape |
| 6 | Sequential twisted PF — $O(T)$ baseline |
| 7 | **Parallel twisted PF** — $O(\log T)$ via prefix scan |
| 8 | Importance-weighted means |
| 9 | Classical baselines: Bootstrap PF + FFBSi |

## Julia vs JAX

| Concept | JAX | Julia |
|---------|-----|-------|
| JIT compilation | `@jit` explicit decorator | Automatic on first call |
| Vectorisation | `vmap` | Broadcasting `.`, `@.` |
| Associative scan | `lax.associative_scan` | Manual loop (sequential); `ThreadsX.accumulate` for parallel |
| GPU | `jax.devices()[0]` | `Metal.MtlArray(...)` (Metal.jl) |
| Random state | Functional key splitting | Mutable `rng` object passed explicitly |

## Packages

- `Random` — standard library; provides `MersenneTwister` and `randn`
- `Statistics` — standard library; provides `mean`, `std`
- `Printf` — standard library; provides C-style `@printf`
- `Plots` — third-party plotting library
- `Metal` — Apple Metal GPU backend for Julia; install with `Pkg.add("Metal")`

`Metal.MtlArray(cpu_array)` moves an array from CPU RAM to the GPU.  
Broadcasting operations on `MtlArray` objects (`.+`, `.*`, `sqrt.()`, etc.) automatically
dispatch GPU kernels, so no explicit kernel code is needed — the same Julia functions
work on both CPU and GPU.

**What runs where in this notebook:**

| Part | Device | Reason |
|------|--------|--------|
| Kalman filter | CPU | Scalar sequential loop; tiny compute |
| Data simulation | CPU | Sequential; tiny compute |
| `noise_tape`, `x0s`, `states` | **GPU** | Large N×T arrays; all ops vectorised over N |
| Twisted PF inner N-loop | **GPU** | Metal kernel per timestep |
| Bootstrap PF / FFBSi | CPU | Inherently sequential; resampling requires scalar indexing |
| Weight normalisation | CPU | `logsumexp` reduction after pulling from GPU |


In [ ]:
using Random
using Statistics
using Printf
using Plots
using Metal


## Model parameters

`const` tells the compiler these values never change, enabling type specialisation and faster code.

| Constant | Value | Meaning |
|----------|-------|---------|
| `PHI` | 0.9 | Auto-regression coefficient. Close to 1 → slowly drifting state. |
| `SIGMA_Q` | 0.5 | Process noise std. Smaller = smoother latent trajectory. |
| `SIGMA_R` | 1.0 | Observation noise std. Larger than `SIGMA_Q` → noisy measurements. |
| `T` | 2000 | Sequence length. On Apple Metal, keep ≤ 4000 to avoid the 10s GPU command-buffer timeout. |
| `N` | 256 | Number of particles. All $N$ trajectories are independent and run simultaneously in the parallel filter. |

In [ ]:
const PHI     = 0.9f0
const SIGMA_Q = 0.5f0
const SIGMA_R = 1.0f0

const T = 2000
const N = 256


## Step 1 — Kalman filter

### What it computes

For a linear-Gaussian SSM the posterior $p(x_t \mid y_{1:t})$ is Gaussian at every step and can be tracked exactly with two scalars: the mean $m_t$ and variance $P_t$. The recursion has two phases:

**Predict** (propagate last posterior through dynamics):
$$m_t^{\text{pred}} = \phi\,m_{t-1}, \qquad P_t^{\text{pred}} = \phi^2 P_{t-1} + \sigma_q^2$$

**Update** (condition on the new observation $y_t$):
$$S_t = P_t^{\text{pred}} + \sigma_r^2 \quad \text{(innovation variance)}$$
$$K_t = P_t^{\text{pred}} / S_t \quad \text{(Kalman gain, between 0 and 1)}$$
$$m_t = m_t^{\text{pred}} + K_t(y_t - m_t^{\text{pred}}), \qquad P_t = (1 - K_t)\,P_t^{\text{pred}}$$

Intuitively, $K_t$ controls how much the observation moves the estimate: $K_t \to 1$ when the observation is very trusted ($\sigma_r \ll \sigma_q$), $K_t \to 0$ when it is very noisy.

### Two roles in this notebook

1. **Ground truth** — `kf_means[t]` is the optimal estimate $\mathbb{E}[x_t \mid y_{1:t}]$; we measure all particle filter RMSEs relative to it.
2. **Twist initialisation** — for a linear-Gaussian model the exact twist function is $\psi_t^*(x_t) = \mathcal{N}(x_t; m_t, P_t)$, so we initialise the twist directly from the KF output. This is not an approximation — it is provably optimal for this model.

### Julia implementation notes

- `Vector{Float64}(undef, T)` allocates an uninitialized length-$T$ vector. `undef` skips zero-filling (safe here because every entry is written in the loop before it is read).
- The type annotation `ys::Vector{Float64}` is optional but helps the compiler generate specialised, faster code for this input type.

In [ ]:
function kalman_filter(ys::Vector{Float32}, phi, sigma_q, sigma_r)
    T = length(ys)
    means = Vector{Float32}(undef, T)
    vars  = Vector{Float32}(undef, T)
    m, P = 0f0, 1f0          # prior: x0 ~ N(0, 1)
    for t in 1:T
        # --- Predict ---
        m_pred = phi * m
        P_pred = phi^2 * P + sigma_q^2
        # --- Update ---
        S = P_pred + sigma_r^2
        K = P_pred / S
        m = m_pred + K * (ys[t] - m_pred)
        P = (1f0 - K) * P_pred
        means[t] = m
        vars[t]  = P
    end
    return means, vars
end


## Step 2 — Twisted proposal and importance weight

### `twisted_step` — sampling from the twisted proposal

The twisted proposal combines the prior transition $p(x_t \mid x_{t-1}) = \mathcal{N}(\phi x_{t-1}, \sigma_q^2)$ with the Gaussian twist $\psi_t(x_t) = \mathcal{N}(\mu_t^\psi, (\tau_t^\psi)^2)$:
$$q_t(x_t \mid x_{t-1}) \propto \mathcal{N}(x_t;\,\phi x_{t-1},\,\sigma_q^2)\cdot\mathcal{N}(x_t;\,\mu_t^\psi,\,(\tau_t^\psi)^2)$$

The product of two Gaussians is another Gaussian. We combine precisions (inverse variances):
$$\lambda_{\text{post}} = \underbrace{\sigma_q^{-2}}_{\texttt{prec\_q}} + \underbrace{(\tau_t^\psi)^{-2}}_{\texttt{prec\_twist}}, \qquad \sigma^2_{\text{post}} = \lambda_{\text{post}}^{-1}$$
$$\mu_{\text{post}} = \sigma^2_{\text{post}}\!\left(\frac{\phi\,x_{t-1}}{\sigma_q^2} + \frac{\mu_t^\psi}{(\tau_t^\psi)^2}\right)$$

`mean_post` is a **precision-weighted average** of the dynamic prediction $\phi x_{t-1}$ and the look-ahead signal $\mu_t^\psi$. A tight twist (small $\tau_t^\psi$) dominates and pulls the proposal toward future observations; a loose twist defers to the dynamics.

The sample is `mean_post + sqrt(var_post) * noise` where `noise` is a **pre-drawn** $\mathcal{N}(0,1)$ value from the tape. Fixing `noise` before the filter runs makes this a **deterministic** map $x_t = f(x_{t-1})$ — the reparametrisation trick — enabling differentiation, Newton's method, and the prefix scan.

`log_tau_twist` stores $\log \tau_t^\psi$ so that `exp(log_tau_twist)` is always positive regardless of any optimisation step that might tune the twist parameters.

---

### `log_weight` — correcting for the biased proposal

Because we sampled from $q_t$ (pulled toward future observations) rather than the true model, we apply an importance weight at every step:
$$\log w_t = \underbrace{\log p(y_t \mid x_t)}_{\text{likelihood}} + \underbrace{\log p(x_t \mid x_{t-1})}_{\text{prior transition}} - \underbrace{\log q_t(x_t \mid x_{t-1})}_{\text{twisted proposal}}$$

Each term is a Gaussian log-density:
- **Likelihood** `log_lik`: rewards $x_t$ close to $y_t$ (uses $\sigma_r$)
- **Prior transition** `log_trans`: penalises $x_t$ far from $\phi x_{t-1}$ (uses $\sigma_q$)
- **Proposal** `log_prop`: subtracts the log-probability under $q_t$, using the same `mean_post` and `var_post` as `twisted_step` — they must match exactly

If the twist were perfect these three terms cancel exactly and all weights are zero in log-space. In practice the residual is small; it is corrected in Step 8.

---

### Julia notes

- `exp(log_tau_twist)` is recomputed in both functions independently so each is self-contained.
- All log-densities are written out explicitly rather than using a distributions library — keeps dependencies minimal.

In [ ]:
function twisted_step(
        x_prev::Float32, noise::Float32,
        mu_twist::Float32, log_tau_twist::Float32,
        phi, sigma_q)
    tau_twist  = exp(log_tau_twist)
    prec_q     = 1f0 / sigma_q^2
    prec_twist = 1f0 / tau_twist^2
    var_post   = 1f0 / (prec_q + prec_twist)
    mean_post  = var_post * (prec_q * phi * x_prev + prec_twist * mu_twist)
    return mean_post + sqrt(var_post) * noise
end

function log_weight(
        x_prev::Float32, x_next::Float32, obs::Float32,
        mu_twist::Float32, log_tau_twist::Float32,
        phi, sigma_q, sigma_r)
    tau_twist  = exp(log_tau_twist)
    prec_q     = 1f0 / sigma_q^2
    prec_twist = 1f0 / tau_twist^2
    var_post   = 1f0 / (prec_q + prec_twist)
    mean_post  = var_post * (prec_q * phi * x_prev + prec_twist * mu_twist)

    log_lik   = -0.5f0 * (obs - x_next)^2 / sigma_r^2 - 0.5f0 * log(2f0 * Float32(π) * sigma_r^2)
    log_trans = -0.5f0 * (x_next - phi * x_prev)^2 / sigma_q^2 - 0.5f0 * log(2f0 * Float32(π) * sigma_q^2)
    log_prop  = -0.5f0 * (x_next - mean_post)^2 / var_post - 0.5f0 * log(2f0 * Float32(π) * var_post)
    return log_lik + log_trans - log_prop
end


## Step 3 — Simulate data

`MersenneTwister(0)` creates a seeded pseudo-random number generator. Unlike JAX (where keys are pure values split explicitly), Julia uses a **mutable RNG object** that advances its internal state on every `randn(rng, ...)` call. Passing `rng` explicitly (rather than using the implicit global) makes the simulation reproducible and thread-safe.

We pre-draw all noise arrays (`q_noise`, `r_noise`) before the simulation loop. This is good practice for reproducibility and mirrors the tape construction in Step 5.

The loop implements:
$$x_t = \phi\,x_{t-1} + \sigma_q\,\varepsilon_t, \qquad y_t = x_t + \sigma_r\,\eta_t$$

`true_xs` is the ground-truth hidden trajectory we are trying to recover. `ys` is the noisy observation sequence that the filter actually sees.

Both arrays are pre-allocated with `Vector{Float64}(undef, T)` and filled in-place. Allocating inside a hot loop would create garbage-collection pressure and slow the code.

In [ ]:
rng = MersenneTwister(0)       # seeded RNG

q_noise = randn(rng, Float32, T)   # process noise  ε_{1:T}
r_noise = randn(rng, Float32, T)   # observation noise  η_{1:T}

true_xs = Vector{Float32}(undef, T)
ys      = Vector{Float32}(undef, T)

x = randn(rng, Float32)        # x0 ~ N(0,1)
for t in 1:T
    x          = PHI * x + SIGMA_Q * q_noise[t]
    true_xs[t] = x
    ys[t]      = x + SIGMA_R * r_noise[t]
end

@printf("Simulated T=%d timesteps\n", T)
@printf("true_xs: mean=%.3f  std=%.3f\n", mean(true_xs), std(true_xs))


## Step 4 — Run the Kalman filter

We pass `ys` through the Kalman filter to get:
- `kf_means[t]` — the posterior mean $m_t = \mathbb{E}[x_t \mid y_{1:t}]$, the optimal estimate of the state
- `kf_vars[t]` — the posterior variance $P_t = \text{Var}[x_t \mid y_{1:t}]$, our remaining uncertainty

These serve as twist parameters in Step 5. `kf_means` also acts as the RMSE benchmark for all particle filter estimates.

In [ ]:
kf_means, kf_vars = kalman_filter(ys, PHI, SIGMA_Q, SIGMA_R)
@printf("KF means: [%.3f, %.3f]\n", minimum(kf_means), maximum(kf_means))

## Step 5 — Build the noise tape and twist parameters

### The noise tape

`noise_tape` is an $N \times T$ matrix of i.i.d. $\mathcal{N}(0,1)$ draws. Entry `noise_tape[n, t]` is the pre-drawn standard normal used when propagating particle $n$ at time $t$.

Fixing all noise upfront makes `twisted_step(x_prev, noise_tape[n,t], ...)` a **deterministic** function of `x_prev`, enabling the prefix scan.

**GPU transfer**: `noise_tape` and `x0s` are created on CPU (Julia's `randn` uses the MersenneTwister), then wrapped with `MtlArray(...)` to move them to Apple Silicon GPU RAM. After this transfer, all broadcasting operations on these arrays dispatch Metal GPU kernels automatically — no explicit kernel code required.

### Twist parameters

`mu_twist[t]` and `log_tau_twist[t]` remain **CPU** `Float32` vectors. They are read as scalar constants inside each time step loop — GPU kernels operate on `noise_tape` and `x0s` while the loop control and twist scalars live on CPU. This is the standard CPU/GPU collaboration pattern: the CPU drives the loop and provides scalars; the GPU executes the heavy N-wide vectorised operations.

### Float32 requirement

Metal (Apple's GPU API) **only supports `Float32`** — it has no hardware `Float64` path. Julia's `MtlArray` enforces this: creating `MtlArray` from a `Float64` array throws an error. All arrays are therefore `Float32` from the start (see data simulation and Kalman filter above).


In [ ]:
# Draw on CPU, then move to GPU with MtlArray
noise_tape = MtlArray(randn(rng, Float32, N, T))   # (N, T) on Metal GPU
x0s        = MtlArray(randn(rng, Float32, N))      # (N,)  on Metal GPU

# Twist parameters stay on CPU — accessed as scalars in the loop
mu_twist      = kf_means                      # (T,) Float32 CPU
log_tau_twist = 0.5f0 .* log.(kf_vars)       # (T,) Float32 CPU

@printf("noise_tape: %d × %d  (GPU: %s)\n", size(noise_tape)..., noise_tape isa MtlArray)
@printf("x0s:        length %d  (GPU: %s)\n", length(x0s), x0s isa MtlArray)


## Step 6 — Sequential twisted particle filter (GPU-vectorised over N)

This is the **O(T) baseline** — we step through $t=1,\ldots,T$ in order. At each step we compute
the twisted proposal for **all $N$ particles simultaneously** using broadcasting on `MtlArray`.

### CPU vs GPU roles

The outer `for t in 1:T` loop runs on CPU. Inside, every arithmetic expression that touches
`xs` or `noise_tape[:, t]` (both `MtlArray{Float32}`) automatically dispatches a Metal GPU kernel
that processes all $N$ elements in parallel. The scalars (`tau_twist`, `prec_q`, `var_post`, etc.)
are CPU `Float32` constants that broadcast against the GPU arrays.

This replaces the old inner `for n in 1:N` loop entirely — Julia's broadcasting is the "vmap".

### Key Julia/Metal notes

**`similar(noise_tape)`** — allocates an uninitialised `MtlArray` of the same shape and type,
on the same device (GPU). This is the Metal-aware equivalent of `Matrix{Float32}(undef, N, T)`.

**`noise_tape[:, t]`** — slices column `t` from the GPU array, returning an `MtlVector{Float32}`.
The slice is a copy (Metal.jl does not support strided views), but the copy is done on-device
without round-tripping through CPU.

**`states[:, t] .= xt`** — in-place column assignment on the GPU. Avoids re-allocating `states`
each iteration.

**`xs = xt`** — rebinds `xs` to the freshly-computed column; the old buffer is eligible for GC.

**Warmup** — Julia compiles a specialised native method (and Metal GPU shader) on the first call.
The warmup call discards that compilation artifact so `@elapsed` measures compiled performance.

**`Array(states_seq)`** — transfers the GPU result back to CPU RAM for `mean` and RMSE computation
(which use standard CPU Julia reductions).


In [ ]:
function run_sequential(x0s, noise_tape, mu_twist, log_tau_twist, phi, sigma_q)
    N, T   = size(noise_tape)
    states = similar(noise_tape)   # (N, T) MtlArray, uninitialised
    xs     = copy(x0s)             # (N,)  working copy on GPU
    for t in 1:T
        tau_twist  = exp(log_tau_twist[t])         # CPU scalar
        prec_q     = 1f0 / sigma_q^2               # CPU scalar
        prec_twist = 1f0 / tau_twist^2             # CPU scalar
        var_post   = 1f0 / (prec_q + prec_twist)   # CPU scalar
        # Broadcast over N particles — dispatches Metal GPU kernel
        xt = var_post .* (prec_q * phi .* xs .+ prec_twist * mu_twist[t]) .+
             sqrt(var_post) .* noise_tape[:, t]
        states[:, t] .= xt      # in-place GPU column write
        xs = xt                 # advance to next step
    end
    return states   # (N, T) MtlArray on GPU
end

_ = run_sequential(x0s, noise_tape, mu_twist, log_tau_twist, PHI, SIGMA_Q)  # warmup / JIT

t_seq     = @elapsed states_seq = run_sequential(x0s, noise_tape, mu_twist, log_tau_twist, PHI, SIGMA_Q)
Metal.synchronize()   # wait for all GPU work to finish before stopping the clock

seq_means = vec(mean(Array(states_seq), dims=1))   # pull to CPU for reduction
rmse_seq  = sqrt(mean((seq_means .- kf_means).^2))
@printf("Sequential twisted PF:  time=%.3fs   RMSE vs Kalman=%.4f\n", t_seq, rmse_seq)


## Step 7 — Parallel twisted particle filter via prefix scan

### Why the step function is affine for D=1

Inspect `twisted_step`. With a Gaussian twist, all terms involving $x_{\text{prev}}$ are linear:
$$x_t^{(n)} = \underbrace{\sigma^2_{\text{post},t}\cdot\frac{\phi}{\sigma_q^2}}_{J_t\;\text{(Jacobian)}}\; x_{t-1}^{(n)} \;+\; \underbrace{\sigma^2_{\text{post},t}\cdot\frac{\mu_t^\psi}{(\tau_t^\psi)^2} + \sigma_{\text{post},t}\cdot\varepsilon_t^{(n)}}_{b_t^{(n)}\;\text{(offset)}}$$

The Jacobian $J_t$ depends only on the twist parameters, **not on $x$**. It is the same scalar for all $N$ particles at time $t$. The offset $b_t^{(n)}$ has a shared part $c_t$ (twist-derived) and a particle-specific part (the pre-drawn noise).

Because the Jacobian is constant (does not depend on $x$), DEER's Newton step is exact in one iteration — the linear system is solved analytically. `affine_coefficients` extracts $J_t$ and $b_t^{(n)}$ for all $t$ and $n$.

### The associative prefix scan

We have $N$ parallel first-order linear recurrences:
$$x_t^{(n)} = J_t\; x_{t-1}^{(n)} + b_t^{(n)}$$

Composing two affine maps $x\mapsto a_1 x+b_1$ then $x\mapsto a_2 x+b_2$ gives $x\mapsto (a_2 a_1)x + (a_2 b_1+b_2)$. This composition is **associative**, enabling a parallel prefix scan over the sequence of pairs $(J_t, b_t)$.

Define elements $(J_t,\,\mathbf{b}_t)\in\mathbb{R}\times\mathbb{R}^N$. The operator is:
$$(a_1,\,\mathbf{b}_1)\oplus(a_2,\,\mathbf{b}_2) = (a_2 a_1,\;\; a_2\,\mathbf{b}_1 + \mathbf{b}_2)$$

After the prefix scan, element $t$ holds $(A_t,\,\mathbf{B}_t)$ where:
$$A_t = \prod_{s=1}^t J_s \qquad \mathbf{B}_t = \sum_{s=1}^t \left(\prod_{r=s+1}^t J_r\right)\mathbf{b}_s$$

The state at every time is then recovered without any further sequential work:
$$x_t^{(n)} = A_t\,x_0^{(n)} + B_t^{(n)}$$

This is exactly the loop in `run_parallel`.

### Why $J_t$ being particle-independent matters

Because $J_t$ is a scalar (same for all $N$ particles), the scan carries one scalar $a$ and one $N$-vector $\mathbf{b}$ per step. The operator updates:
- Scalar: `as_acc[t] = Js[t] * as_acc[t-1]` — one multiply
- Vector: `Bs_acc[:, t] .= Js[t] .* Bs_acc[:, t-1] .+ Bs[:, t]` — two vectorised ops over $N$ elements

All $N$ particles are updated simultaneously. On a GPU with $N$ threads, the vector update is $O(1)$ work per thread per step.

### Broadcasting shape note

The recovery line:
```julia
states = reshape(x0s, N, 1) .* reshape(as_acc, 1, T) .+ Bs_acc
```
uses explicit reshapes:
- `reshape(x0s, N, 1)` → `(N, 1)` column
- `reshape(as_acc, 1, T)` → `(1, T)` row
- their product broadcasts to `(N, T)` (outer product)
- `+ Bs_acc` is element-wise `(N, T) + (N, T)`

Without the reshapes, Julia cannot unambiguously align a length-$N$ vector with a length-$T$ vector and raises a `DimensionMismatch`.

### Sequential vs parallel implementation

The loop here runs sequentially. The inner `.=` operations vectorise via SIMD (all $N$ particles update in parallel on the CPU's vector units), giving some speedup. For full $O(\log T)$ depth:
- **Multi-core**: replace the loop with `ThreadsX.accumulate`
- **Apple Silicon Metal**: wrap arrays in `Metal.MtlArray` and use GPU-parallel operations
### Metal GPU execution

In this implementation, `Bs_acc` is a GPU `MtlArray`. The loop body:
```julia
Bs_acc[:, t] .= Js[t] .* Bs_acc[:, t-1] .+ Bs[:, t]
```
dispatches two Metal kernels per step (scale + add, then store). All $N$ particles are updated in parallel by the GPU at each of the $T$ sequential steps.

`as_acc` remains a CPU `Float32` vector — it is a single scalar per step and is not worth moving to GPU.

The final `MtlArray(as_acc)` call transfers the accumulated scalar multipliers to GPU for the $O(N \times T)$ broadcast that recovers all particle states at once.


In [ ]:
function affine_coefficients(noise_tape, mu_twist, log_tau_twist, phi, sigma_q)
    N, T       = size(noise_tape)
    Js         = Vector{Float32}(undef, T)    # (T,) CPU scalar per timestep
    Bs         = similar(noise_tape)          # (N, T) GPU — inherits MtlArray type
    prec_q     = 1f0 / sigma_q^2
    for t in 1:T
        tau_twist  = exp(log_tau_twist[t])
        prec_twist = 1f0 / tau_twist^2
        var_post   = 1f0 / (prec_q + prec_twist)
        Js[t]      = var_post * prec_q * phi
        c          = var_post * prec_twist * mu_twist[t]
        # Broadcast over N on GPU
        Bs[:, t]  .= c .+ sqrt(var_post) .* noise_tape[:, t]
    end
    return Js, Bs
end

function run_parallel(x0s, noise_tape, mu_twist, log_tau_twist, phi, sigma_q)
    N, T = size(noise_tape)

    Js, Bs = affine_coefficients(noise_tape, mu_twist, log_tau_twist, phi, sigma_q)

    # Prefix scan:
    #   as_acc  — scalar CPU (one multiply per step)
    #   Bs_acc  — MtlArray GPU (N-wide scale+add per step)
    as_acc  = Vector{Float32}(undef, T)
    Bs_acc  = similar(Bs)                   # (N, T) GPU
    as_acc[1]      = Js[1]
    Bs_acc[:, 1]  .= Bs[:, 1]
    for t in 2:T
        as_acc[t]     = Js[t] * as_acc[t-1]                      # CPU scalar update
        Bs_acc[:, t] .= Js[t] .* Bs_acc[:, t-1] .+ Bs[:, t]     # GPU N-vector update
    end

    # Recover states: x_t^(n) = A_t * x0^(n) + B_t^(n)
    # Move as_acc to GPU for the final (N,T) broadcast
    as_gpu = MtlArray(as_acc)                          # (T,) → GPU
    states = reshape(x0s, N, 1) .* reshape(as_gpu, 1, T) .+ Bs_acc
    return states   # (N, T) MtlArray
end

_ = run_parallel(x0s, noise_tape, mu_twist, log_tau_twist, PHI, SIGMA_Q)  # warmup

t_par = @elapsed states_par = run_parallel(x0s, noise_tape, mu_twist, log_tau_twist, PHI, SIGMA_Q)
Metal.synchronize()

par_means = vec(mean(Array(states_par), dims=1))
rmse_par  = sqrt(mean((par_means .- kf_means).^2))
@printf("Parallel twisted PF:    time=%.3fs   RMSE vs Kalman=%.4f\n", t_par, rmse_par)
@printf("Speedup: %.1fx\n", t_seq / t_par)


## Step 8 — Importance-weighted means

The unweighted particle mean (`par_means`) treats all $N$ particles equally. This is good when the twist is nearly perfect (weights are near-uniform), but we can do better by applying the full importance weights.

### Cumulative log-weight

The incremental weight $\log w_t^{(n)}$ measures how consistent particle $n$'s step at time $t$ is with the true model. Cumulating these:
$$\log W_t^{(n)} = \sum_{s=1}^t \log w_s^{(n)}$$

`cumsum(log_ws, dims=2)` computes this left-to-right prefix sum over the time dimension (columns).

### Log-sum-exp

Normalising weights requires computing $Z_t = \sum_n \exp(\log W_t^{(n)})$. Naive exponentiation overflows for large weights and underflows for small ones. The log-sum-exp trick shifts by the maximum before exponentiating:
$$\log Z_t = m_t + \log\sum_n e^{\log W_t^{(n)} - m_t}, \qquad m_t = \max_n \log W_t^{(n)}$$

Subtracting $m_t$ keeps all arguments in $(-\infty, 0]$, preventing overflow. This is `logsumexp`.

### Broadcasting in normalisation

`cum_log_ws .- log_norm'` subtracts the `(T,)` vector `log_norm` from the `(N,T)` matrix `cum_log_ws`. The transpose `log_norm'` produces a `(1,T)` row that broadcasts across all $N$ rows — each column $t$ gets the same normalisation constant $\log Z_t$.

### Weighted mean

`norm_ws .* states_par` is element-wise `(N,T) .* (N,T)`. Summing over `dims=1` reduces across particles at each $t$, giving the weighted estimate $\hat{x}_t = \sum_n W_t^{(n)} x_t^{(n)}$.

In [ ]:
function logsumexp(v::AbstractVector)
    m = maximum(v)
    return m + log(sum(exp.(v .- m)))
end

# Vectorised importance weights — inner N loop replaced by GPU broadcasting
function compute_log_weights(x0s, states, ys, mu_twist, log_tau_twist, phi, sigma_q, sigma_r)
    N, T   = size(states)
    log_ws = similar(states)   # (N, T) MtlArray
    for t in 1:T
        x_prev = t == 1 ? x0s : states[:, t-1]   # (N,) GPU
        x_next = states[:, t]                     # (N,) GPU
        obs    = ys[t]                            # CPU scalar

        tau_twist  = exp(log_tau_twist[t])
        prec_q     = 1f0 / sigma_q^2
        prec_twist = 1f0 / tau_twist^2
        var_post   = 1f0 / (prec_q + prec_twist)
        mean_post  = var_post .* (prec_q .* phi .* x_prev .+ prec_twist * mu_twist[t])

        # All three terms broadcast over N on GPU
        log_lik   = (-0.5f0) .* (obs .- x_next).^2 ./ sigma_r^2 .-
                    0.5f0 * log(2f0 * Float32(π) * sigma_r^2)
        log_trans = (-0.5f0) .* (x_next .- phi .* x_prev).^2 ./ sigma_q^2 .-
                    0.5f0 * log(2f0 * Float32(π) * sigma_q^2)
        log_prop  = (-0.5f0) .* (x_next .- mean_post).^2 ./ var_post .-
                    0.5f0 * log(2f0 * Float32(π) * var_post)

        log_ws[:, t] .= log_lik .+ log_trans .- log_prop
    end
    return log_ws   # (N, T) MtlArray
end

log_ws = compute_log_weights(x0s, states_par, ys, mu_twist, log_tau_twist, PHI, SIGMA_Q, SIGMA_R)

# Pull to CPU for the sequential reductions (logsumexp, cumsum)
log_ws_cpu     = Array(log_ws)                                    # (N, T) CPU
cum_log_ws     = cumsum(log_ws_cpu, dims=2)                       # (N, T)
log_norm       = [logsumexp(cum_log_ws[:, t]) for t in 1:T]      # (T,)
norm_ws        = exp.(cum_log_ws .- log_norm')                    # (N, T), cols sum to 1
states_par_cpu = Array(states_par)                                # (N, T) CPU
weighted_par_means = vec(sum(norm_ws .* states_par_cpu, dims=1)) # (T,)

rmse_par_w = sqrt(mean((weighted_par_means .- kf_means).^2))
@printf("Parallel PF (weighted):                 RMSE vs Kalman=%.4f\n", rmse_par_w)


## Step 9 — Classical baselines

### Bootstrap particle filter

The bootstrap PF uses no look-ahead — it propagates from the blind prior:
$$x_t^{(n)} \sim p(x_t \mid x_{t-1}^{(n)}) = \mathcal{N}(\phi x_{t-1}^{(n)},\, \sigma_q^2)$$
then weights by the likelihood:
$$\log w_t^{(n)} = -\frac{(y_t - x_t^{(n)})^2}{2\sigma_r^2} + \text{const}$$

Because the proposal ignores $y_t$, particles spread blindly. Many will have near-zero weight. We prevent weight collapse by **systematic resampling** when the effective sample size (ESS) drops below $N/2$:
$$\text{ESS}_t = \frac{1}{\sum_n (w_t^{(n)})^2} \in [1, N]$$

Resampling duplicates high-weight particles and discards low-weight ones, but introduces a coupling between timesteps that makes parallelism impossible.

The output is a **filter** (uses only $y_{1:t}$), so its RMSE will be somewhat worse than the smoothers.

### Systematic resampling

Draw one uniform $u \sim U[0, 1/N]$, then place $N$ equally-spaced points $u,\; u+1/N,\; \ldots$ Each point selects the particle whose cumulative weight first exceeds it. This guarantees every particle with weight $>1/N$ is selected at least once, giving much lower variance than drawing $N$ independent multinomial samples.

### `@.` macro

`log_ws = @. -0.5 * (ys[t] - xs)^2 / sigma_r^2` broadcasts every operation automatically — equivalent to writing `.` after every operator and function call. Saves visual clutter for element-wise expressions.

---

### Forward-filter backward-smoother (FFBSi)

FFBSi uses **all** observations $y_{1:T}$ to smooth state estimates. It has two passes:

**Forward**: already done by the bootstrap PF — produces filtered clouds $\{x_t^{(n)}, w_t^{(n)}\}$.

**Backward**: sample $M$ complete trajectories. For trajectory $m$:
1. Sample terminal state $x_T^{(m)}$ from the $T$-step filtered distribution
2. Step backward $t = T-1,\ldots,1$: sample $x_t^{(m)}$ from filtered particles weighted by:
$$w_{t|t+1}^{(n)} \propto w_t^{(n)} \cdot p(x_{t+1}^{(m)} \mid x_t^{(n)}) = w_t^{(n)}\cdot\mathcal{N}(x_{t+1}^{(m)};\,\phi x_t^{(n)},\,\sigma_q^2)$$

The backward weight combines:
- $w_t^{(n)}$ — how good was particle $n$ given the **past** data $y_{1:t}$
- $p(x_{t+1}^{(m)} \mid x_t^{(n)})$ — how likely is the already-sampled next state given particle $n$

This integrates future information $y_{t+1:T}$ (encoded in $x_{t+1}^{(m)}$) into the estimate at time $t$. The result is a smoother, not just a filter.

Cost: $O(MNT)$ — each of $M$ trajectories requires $T$ steps each touching all $N$ particles. Expensive, but classically gold-standard.

In [ ]:
function systematic_resample(weights::AbstractVector{Float32}, rng)
    N    = length(weights)
    cumw = cumsum(weights)
    u    = rand(rng, Float32) / N
    j    = 1
    idx  = Vector{Int}(undef, N)
    for i in 1:N
        while u > cumw[j]; j += 1; end
        idx[i] = j
        u += 1f0 / N
    end
    return idx
end

function bootstrap_pf(ys, x0s, phi, sigma_q, sigma_r, rng)
    N             = length(x0s)
    T             = length(ys)
    filt_means    = Vector{Float32}(undef, T)
    all_particles = Matrix{Float32}(undef, N, T)
    all_log_ws    = Matrix{Float32}(undef, N, T)
    xs = copy(x0s)
    for t in 1:T
        xs .= phi .* xs .+ sigma_q .* randn(rng, Float32, N)

        log_ws  = @. -0.5f0 * (ys[t] - xs)^2 / sigma_r^2
        log_ws .-= logsumexp(log_ws)
        ws = exp.(log_ws)

        filt_means[t]       = sum(ws .* xs)
        all_particles[:, t] = xs
        all_log_ws[:, t]    = log_ws

        ess = 1f0 / sum(ws.^2)
        if ess < N / 2
            idx = systematic_resample(ws, rng)
            xs .= xs[idx]
            all_log_ws[:, t] .= -log(Float32(N))
        end
    end
    return filt_means, all_particles, all_log_ws
end

bpf_rng = MersenneTwister(1)
x0s_bpf = randn(bpf_rng, Float32, N)
_ = bootstrap_pf(ys, x0s_bpf, PHI, SIGMA_Q, SIGMA_R, bpf_rng)  # warmup

bpf_rng = MersenneTwister(1); x0s_bpf = randn(bpf_rng, Float32, N)
t_bpf = @elapsed begin
    bpf_means, bpf_parts, bpf_log_ws = bootstrap_pf(
        ys, x0s_bpf, PHI, SIGMA_Q, SIGMA_R, bpf_rng)
end
rmse_bpf = sqrt(mean((bpf_means .- kf_means).^2))
@printf("Bootstrap PF (filter):  time=%.3fs   RMSE vs Kalman=%.4f\n", t_bpf, rmse_bpf)


In [ ]:
function ffbsi_smoother(all_particles, all_log_ws, phi, sigma_q, M, rng)
    N, T  = size(all_particles)
    trajs = Matrix{Float32}(undef, M, T)

    ws_T  = exp.(all_log_ws[:, T])
    ws_T ./= sum(ws_T)

    for m in 1:M
        idx         = findfirst(cumsum(ws_T) .>= rand(rng, Float32))
        idx         = isnothing(idx) ? N : idx
        trajs[m, T] = all_particles[idx, T]

        for t in T-1:-1:1
            x_next = trajs[m, t+1]
            log_bw  = all_log_ws[:, t] .-
                      0.5f0 .* ((x_next .- phi .* all_particles[:, t]).^2) ./ sigma_q^2
            log_bw .-= logsumexp(log_bw)
            bw       = exp.(log_bw)

            idx2        = findfirst(cumsum(bw) .>= rand(rng, Float32))
            idx2        = isnothing(idx2) ? N : idx2
            trajs[m, t] = all_particles[idx2, t]
        end
    end
    return trajs
end

ffbsi_rng = MersenneTwister(2)
M_back    = N
_ = ffbsi_smoother(bpf_parts, bpf_log_ws, PHI, SIGMA_Q, M_back, ffbsi_rng)  # warmup

ffbsi_rng = MersenneTwister(2)
t_bps = @elapsed bps_trajs = ffbsi_smoother(
    bpf_parts, bpf_log_ws, PHI, SIGMA_Q, M_back, ffbsi_rng)
bps_means = vec(mean(bps_trajs, dims=1))
rmse_bps  = sqrt(mean((bps_means .- kf_means).^2))
@printf("Bootstrap PS (FFBSi):   time=%.3fs   RMSE vs Kalman=%.4f  (M=%d)\n",
        t_bps, rmse_bps, M_back)


## Results

### What to look for

**RMSE parity between sequential and parallel twisted PF** — both use the same noise tape and twist parameters, so their particle means should be nearly identical. Any difference is due to floating-point arithmetic order.

**Smoother RMSE < filter RMSE** — FFBSi and both twisted PF variants use future observations ($y_{t:T}$), giving lower RMSE than the bootstrap PF filter (which uses only $y_{1:t}$).

**Twisted PF RMSE ≈ FFBSi RMSE** — with the optimal Kalman twist the twisted PF should match FFBSi quality, at a fraction of the cost.

**Speedup** — on CPU with the sequential scan loop, the speedup comes from SIMD vectorisation over $N$ particles. On a GPU the speedup would grow with $T$ as more steps run in parallel.

In [ ]:
@printf("\nT=%d, N=%d particles\n", T, N)
@printf("%-35s %10s %12s\n", "Method", "Time (s)", "RMSE vs KF")
println("-"^60)
@printf("%-35s %10.3f %12.4f\n", "Sequential twisted PF",       t_seq,  rmse_seq)
@printf("%-35s %10.3f %12.4f\n", "Parallel twisted PF (scan)",  t_par,  rmse_par)
@printf("%-35s %10s %12.4f\n",   "  (importance-weighted)",     "",     rmse_par_w)
@printf("%-35s %10.3f %12.4f\n", "Bootstrap PF (filter)",       t_bpf,  rmse_bpf)
@printf("%-35s %10.3f %12.4f\n", "Bootstrap PS (FFBSi)",        t_bps,  rmse_bps)
@printf("\nSpeedup (seq\u2192parallel):   %.1fx\n", t_seq / t_par)
@printf("Speedup (FFBSi\u2192parallel):  %.1fx\n",  t_bps / t_par)

## Plots

**Top panel** — sequential twisted PF vs Kalman. The Kalman mean (green) and sequential PF mean (blue dashed) should both track `true_xs` (black) closely. The noisy observations (grey dots) illustrate how much signal is in $y_t$.

**Bottom panel** — parallel twisted PF vs Kalman. Because both twisted PF variants use the same noise tape and twist, their particle means should be visually indistinguishable. The title reports the wall-clock speedup.

`@sprintf` formats floats into strings at plot-time, embedding the RMSE and speedup values directly into the legend and title.

In [ ]:
t_idx = 1:T

p1 = plot(t_idx, true_xs,  color=:black, lw=1, alpha=0.6, label="true state",
          title="Sequential twisted PF vs Kalman", ylabel="state")
scatter!(p1, t_idx, ys, color=:grey, ms=1, alpha=0.2,
         label="observations", markerstrokewidth=0)
plot!(p1, t_idx, kf_means,  color=:green, lw=1.5, label="Kalman mean")
plot!(p1, t_idx, seq_means, color=:blue,  lw=1.5, ls=:dash,
      label=@sprintf("seq twisted PF (RMSE=%.3f)", rmse_seq))

p2 = plot(t_idx, true_xs,  color=:black, lw=1, alpha=0.6, label="true state",
          title=@sprintf("Parallel twisted PF vs Kalman — %.1fx speedup", t_seq/t_par),
          xlabel="timestep", ylabel="state")
plot!(p2, t_idx, kf_means,  color=:green, lw=1.5, label="Kalman mean")
plot!(p2, t_idx, par_means, color=:red,   lw=1.5, ls=:dash,
      label=@sprintf("parallel twisted PF (RMSE=%.3f)", rmse_par))

fig = plot(p1, p2, layout=(2,1), size=(900,500), legend=:outertopright)
savefig(fig, "twisted_pf_linear_gaussian_julia.png")
fig
